# 01c Actual Dataset, Time Split

This notebook runs Experiment 3 for the journal article: PyNRPF is trained and evaluated on the actual manually labelled dataset using the conference-paper time split.

The intent is to test temporal generalization on real wrong-positive readings. This is the closest extension of the conference-paper time split, but using manually labelled actual data rather than synthetic labels.

By default the notebook runs in smoke mode so that dataset loading, date windows, and output wiring can be checked before full model training.

## Databricks Dependency Setup

Run this cell before the import/setup cells when executing the notebook in Databricks. The helper module imports `yaml`, which is provided by the `PyYAML` package. Keeping the install cell inside every experiment notebook makes each notebook runnable on a fresh Databricks cluster without depending on cluster-level library setup.


In [ ]:
%pip install PyYAML


## Imports And Path Setup

This section locates the journal article folder and imports the shared experiment helpers.

The path search avoids local absolute paths, which makes the notebook easier to run from an IDE, terminal, or future Databricks-style environment.

If this cell fails, verify the repository layout and confirm `_experiment_helpers.py` is present in `publication/2_journal_article/notebooks/`.

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
helper_dir = None
for candidate in [start, *start.parents]:
    direct = candidate if candidate.name == "notebooks" else candidate / "publication" / "2_journal_article" / "notebooks"
    if (direct / "_experiment_helpers.py").exists():
        helper_dir = direct
        break
if helper_dir is None:
    raise RuntimeError("Could not locate _experiment_helpers.py")
sys.path.insert(0, str(helper_dir))

from _experiment_helpers import (
    dataset_summary,
    experiment_output_dir,
    find_article_root,
    load_config,
    load_dataset,
    run_time_split_experiment,
    time_masks,
)

ARTICLE_ROOT = find_article_root(start)
NOTEBOOK_NAME = "01c_actual_time_split.ipynb"
EXPERIMENT_ID = "experiment_3_actual_time_split"
DATASET_KEY = "actual"
ARTICLE_ROOT

## Load YAML Config

Editable constants live in `config/experiment_config.yaml` rather than being hard-coded in this notebook.

Important values loaded from YAML include dataset paths, split dates, active methods, M8 hyperparameters and thresholds, M7 threshold settings, resume behavior, evaluation rules, and output folder names.

`methods.enabled` is the only place to choose methods. Use `["m7_dtr"]` to run M7 only, or `["m8_xgb", "m7_dtr"]` to run both. When `m8_xgb` is absent, the helper does not train M8 at all.

The most important safety flag is `execution.run_full_experiment`. When it is `false`, the notebook performs smoke validation only. When it is `true`, it can train models and write full CSV outputs. Resume behavior is controlled by `resume.skip_completed`; completed fold/method tasks are skipped unless `execution.overwrite_outputs` is set to `true`.


In [ ]:
cfg = load_config(ARTICLE_ROOT)
active_methods = cfg["methods"]["enabled"]
print("Config:", cfg["_config_path"])
print("Run full experiment:", cfg["execution"]["run_full_experiment"])
print("Enabled methods:", active_methods)
print("M8 enabled:", "m8_xgb" in active_methods)
print("M8 behavior:", "train only for incomplete M8 tasks" if "m8_xgb" in active_methods else "skip all M8 training")
print("Resume skip completed:", cfg.get("resume", {}).get("skip_completed", False))
print("Overwrite outputs:", cfg["execution"].get("overwrite_outputs", False))
print("Output folder:", experiment_output_dir(ARTICLE_ROOT, cfg, EXPERIMENT_ID))


## Load And Validate Dataset

This reads the processed actual CSV, validates schema and labels, and prints a compact summary.

The actual dataset contains anonymized site IDs (`act_A` to `act_H`) and manually derived labels. The helper recomputes `label_day` from `label_interval` so stale day labels cannot silently affect results.

The expected actual dataset has 8 stations and spans November 2021 through September 2024. Null `net_load_MW` and `solar_MW` rows are retained, matching the dataset-preparation decision.

In [ ]:
df = load_dataset(ARTICLE_ROOT, cfg, DATASET_KEY)
dataset_summary(df, cfg, DATASET_KEY)

## Define Time Split

The train/test masks are driven by YAML split dates.

The train partition covers `2021-11-01` through `2023-09-30`. The test partition covers `2023-10-01` through `2024-09-30`.

If the row counts differ from expectation, inspect timestamp parsing and the date range in the processed actual CSV.

In [ ]:
train_mask, test_mask = time_masks(df, cfg)
print("Train rows:", int(train_mask.sum()))
print("Test rows:", int(test_mask.sum()))

## Method Helpers

M8 and M7 execution is implemented in the shared helper module to keep this notebook readable while keeping all experiment notebooks consistent.

Active methods come from `methods.enabled` in YAML. The notebook does not maintain a second method list. If the list is only `["m7_dtr"]`, M7 runs independently and no M8 feature building or XGBoost training is attempted. If `m8_xgb` is enabled, M8 training happens only for fold/method tasks that are missing a complete checkpoint.

M8 is the trainable two-stage XGBoost method: first a day-level classifier, then an interval-level classifier inside candidate days. M7 is the deterministic threshold-rule baseline and does not train.

Both methods use fixed conference-paper settings from YAML. This is deliberate: the journal experiments compare generalization settings, not retuned parameter sets.


In [ ]:
active_methods = cfg["methods"]["enabled"]
print("Enabled methods:", active_methods)
print("M8 enabled:", "m8_xgb" in active_methods)
print("M8 behavior:", "train only for incomplete M8 tasks" if "m8_xgb" in active_methods else "skip all M8 training")
print("M7 enabled:", "m7_dtr" in active_methods)
print("Resume skip completed:", cfg.get("resume", {}).get("skip_completed", False))
print("Overwrite outputs:", cfg["execution"].get("overwrite_outputs", False))
print("M8 thresholds:", cfg["m8_xgb"]["xgb1_day"]["threshold"], cfg["m8_xgb"]["xgb2_timestamp"]["threshold"])


## Run Experiment

Smoke mode writes a manifest only and skips XGBoost training.

Full mode trains M8 on the actual train period, evaluates M8 and M7 on both train and test partitions, and writes metrics plus prediction CSV files.

Use smoke mode first whenever the dataset or config changes.

In full mode, each fold/method task writes three checkpoint files as soon as that task completes: a prediction CSV under `predictions/`, a per-task metrics CSV under `metrics/`, and a completion YAML under `status/`. The completion marker is written last, so interrupted or failed tasks rerun cleanly.

On rerun, completed tasks are skipped when `resume.skip_completed: true` and `execution.overwrite_outputs: false`. The notebook also rebuilds `metrics_summary.csv`, `fold_metrics.csv`, and `manifest.yaml` from completed task files each time, so partial output remains inspectable even if a long run stops halfway.


In [ ]:
run_time_split_experiment(ARTICLE_ROOT, cfg, DATASET_KEY, EXPERIMENT_ID, NOTEBOOK_NAME)

## Quick Review

Use this final cell to confirm where the notebook wrote outputs.

After a smoke run, expect `manifest.yaml` plus created `predictions/`, `metrics/`, and `status/` folders. After a full or partial run, inspect `status/` first to see which fold/method tasks completed, then inspect the per-task metrics and prediction CSVs.

`metrics_summary.csv`, `fold_metrics.csv`, and `manifest.yaml` are rebuilt from completed task files on every run. If results look wrong later, start debugging by checking the status YAML for the affected fold/method, then compare the manifest settings against `experiment_config.yaml`.


In [ ]:
print("Review output folder:", experiment_output_dir(ARTICLE_ROOT, cfg, EXPERIMENT_ID))